In [2]:
import os
import csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
import networkx as nx
from skimage.morphology import skeletonize
from sklearn.decomposition import PCA

# Paths
root_dir = 'final_roots_new'
output_csv = 'primary_roots3_info.csv'
os.makedirs('primary_root_visuals_3', exist_ok=True)

# List root mask files
root_files = sorted([f for f in os.listdir(root_dir) if f.endswith('.png')])
print(f"Found {len(root_files)} root images.")

# Write CSV
with open(output_csv, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['Root File', 'Length (pixels)', 'Tip Y', 'Tip X'])

    for filename in root_files:
        print(f"\nProcessing: {filename}")
        path = os.path.join(root_dir, filename)

        # Load & binarize
        mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        binary = (mask > 127).astype(np.uint8)

        # Morphological cleaning
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # Skeletonize
        skeleton = skeletonize(binary).astype(np.uint8)

        # Build graph
        G = nx.Graph()
        h, w = skeleton.shape
        for y in range(h):
            for x in range(w):
                if skeleton[y, x]:
                    for dy in [-1, 0, 1]:
                        for dx in [-1, 0, 1]:
                            if dx == 0 and dy == 0:
                                continue
                            ny, nx_ = y + dy, x + dx
                            if 0 <= ny < h and 0 <= nx_ < w and skeleton[ny, nx_]:
                                G.add_edge((y, x), (ny, nx_))

        if len(G.nodes()) < 2:
            print("Not enough skeleton points.")
            writer.writerow([filename, 'TOO_SMALL', '', ''])
            continue

        # Keep only largest connected component
        if not nx.is_connected(G):
            largest_cc = max(nx.connected_components(G), key=len)
            G = G.subgraph(largest_cc).copy()

        # Find endpoints
        endpoints = [n for n in G.nodes() if G.degree(n) == 1]
        if len(endpoints) < 2:
            print("Not enough endpoints.")
            writer.writerow([filename, 'NO_ENDPOINTS', '', ''])
            continue

        # --- Try PCA-based method first ---
        primary_path = []
        try:
            coords = np.array(list(G.nodes()))
            pca = PCA(n_components=2)
            pca.fit(coords)
            axis = pca.components_[0]
            projections = [np.dot(c, axis) for c in endpoints]
            base = endpoints[np.argmin(projections)]
            tip = endpoints[np.argmax(projections)]
            primary_path = nx.shortest_path(G, base, tip)
        except Exception as e:
            print("PCA method failed:", e)

        # --- Fallback: longest path between any two endpoints ---
        if not primary_path:
            max_len = 0
            for i in range(len(endpoints)):
                for j in range(i + 1, len(endpoints)):
                    try:
                        path = nx.shortest_path(G, endpoints[i], endpoints[j])
                        if len(path) > max_len:
                            primary_path = path
                            max_len = len(path)
                    except nx.NetworkXNoPath:
                        continue

        # --- If still no path, skip ---
        if not primary_path:
            print("No valid path found.")
            writer.writerow([filename, 'NO_PATH', '', ''])
            continue

        # Tip and length
        tip_yx = primary_path[-1]
        length = len(primary_path)

        # Save to CSV
        writer.writerow([filename, length, tip_yx[0], tip_yx[1]])

        # Visualization
        vis = np.stack([skeleton * 255] * 3, axis=-1).astype(np.uint8)
        for (y, x) in primary_path:
            vis[y, x] = [255, 0, 0]
        cv2.imwrite(os.path.join('primary_root_visuals_3', filename), vis)

        print(f"Length: {length} | Tip: {tip_yx}")


Found 4 root images.

Processing: plant_0.png
Length: 1205 | Tip: (1215, 320)

Processing: plant_1.png
Length: 132 | Tip: (127, 1)

Processing: plant_2.png
Length: 1448 | Tip: (1443, 215)

Processing: plant_3.png
Length: 810 | Tip: (883, 237)
